In [122]:
!pip install awswrangler==3.12.0 evidently==0.7.8 -q

In [123]:
from monitoring_utils import get_username

# GLOBAL VARIABLES

In [124]:
USERNAME = get_username()

# Evidently workspace
workspace = "https://8kgtpeaxtj.us-east-1.awsapprunner.com"  # ✔ tu instancia actual

# Training variables
TARGET_COL = "attrition"         # ✔ tu columna objetivo real
SEED = 42
TRAIN_SPLIT = 0.7

FEATURES = [
    "flg_bancarizado",
    "rang_ingreso",
    "edad",
    "antiguedad"
    # puedes añadir más columnas reales aquí si están en tu CSV
]

# PROJECT CREATION OR LOADING

In [125]:
from evidently.ui.workspace import RemoteWorkspace

In [126]:
ws = RemoteWorkspace(workspace)

In [127]:
project_name = "Monitoreo de Sistema de Detección de Fraude en TC"
project_description = "Este proyecto monitorea tanto la data como el modelo del sistema de ML de fraude" 
project = ws.search_project(project_name)
if project:
    project = project[0]
    print("Project already exists")
else:
    print("Creating new project")
    project = ws.create_project(project_name)
    project.description = project_description
    project.save()
#ws.delete_project(project.id)

Project already exists


# DATA PULL

In [128]:
import awswrangler as wr
from evidently.presets import DataSummaryPreset
from evidently import Dataset
from evidently import DataDefinition
from evidently import Report

In [129]:
import awswrangler as wr

# Ruta de tu archivo CSV en S3
s3_path = "s3://s3-mlflow-artifacts-mlflow-tracking-server-grupo5-01ggz/sagemaker/clients-attrition/grupo5/data_train/trained_clientes_pipeline.csv"

# Cargar el dataset desde S3
df = wr.s3.read_csv(s3_path)

# Mostrar las primeras filas para verificar
df.head()

,id_cliente,codmes_cliente,flg_bancarizado,rang_ingreso,flag_lima_provincia,edad,antiguedad,attrition,codmes_requerimiento,tipo_requerimiento2,dictamen,producto_servicio_2,submotivo_2
0,35653,201208,1,Rang_ingreso_06,Lima,25.0,6.0,0,201208.0,Reclamo,NO PROCEDE,Producto 20,Submotivo 145
1,66575,201208,1,Rang_ingreso_03,Provincia,27.0,0.0,0,NaN,NaN,NaN,NaN,NaN
2,56800,201208,1,Rang_ingreso_01,Provincia,34.0,4.0,0,202405.0,Reclamo,PROCEDE PARCIAL,Producto 07,Submotivo 125
3,8410,201208,1,Rang_ingreso_04,Provincia,63.0,5.0,0,202404.0,Solicitud,NO PROCEDE,Producto 20,Submotivo 157
4,6853,201208,1,NaN,Lima,25.0,0.0,0,NaN,NaN,NaN,NaN,NaN


In [130]:
df.dropna(inplace=True)
df[TARGET_COL] = df[TARGET_COL].astype(str)

# 2) variables categóricas
df["flg_bancarizado"] = df["flg_bancarizado"].astype(str).astype("category")
# (si "rang_ingreso" es un rango discreto como 'A', 'B', 'C', también categórico)
df["rang_ingreso"] = df["rang_ingreso"].astype(str).astype("category")

# 3) variables numéricas
df["edad"]        = df["edad"].astype("int16")     # o "float32" si prefieres
df["antiguedad"]  = df["antiguedad"].astype("int16")

In [131]:
df.head(3)

,id_cliente,codmes_cliente,flg_bancarizado,rang_ingreso,flag_lima_provincia,edad,antiguedad,attrition,codmes_requerimiento,tipo_requerimiento2,dictamen,producto_servicio_2,submotivo_2
0,35653,201208,1,Rang_ingreso_06,Lima,25,6,0,201208.0,Reclamo,NO PROCEDE,Producto 20,Submotivo 145
2,56800,201208,1,Rang_ingreso_01,Provincia,34,4,0,202405.0,Reclamo,PROCEDE PARCIAL,Producto 07,Submotivo 125
3,8410,201208,1,Rang_ingreso_04,Provincia,63,5,0,202404.0,Solicitud,NO PROCEDE,Producto 20,Submotivo 157


In [132]:
raw_definition = DataDefinition(
    id_column      = "id_cliente",       
    timestamp      = "codmes_cliente",   

    categorical_columns = [
        TARGET_COL,               # "attrition"
        "flg_bancarizado",
        "rang_ingreso"
    ],

    numerical_columns = [
        "edad",
        "antiguedad"
    ]
)
# https://docs.evidentlyai.com/docs/library/data_definition

In [133]:
dataset = Dataset.from_pandas(df, data_definition=raw_definition)

In [134]:
data_summary_report = Report([DataSummaryPreset()], tags=["Data summary"])
data_summary_run = data_summary_report.run(dataset, None)
data_summary_run.save_html("data_summary.html")
data_summary_ref = ws.add_run(project.id, data_summary_run)
print(f"Report's link: {data_summary_ref.url}")
print(f"Report's id: {data_summary_ref.id}")
#ws.delete_run(project.id, data_summary_ref.id)

Report's link: https://8kgtpeaxtj.us-east-1.awsapprunner.com/projects/0197c00c-c2c1-76a3-be64-3ee4604ef948/reports/0197c294-df56-791c-8169-722e773afb28
Report's id: 0197c294-df56-791c-8169-722e773afb28


# MODEL TRAINING

In [135]:
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
import pandas as pd
from evidently import BinaryClassification

In [136]:
X = df[FEATURES]
y = df[TARGET_COL].astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    train_size=TRAIN_SPLIT,
                                                    random_state=SEED)

In [137]:
xgb = XGBClassifier(enable_categorical=True)
xgb.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric=None, feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None,
              num_parallel_tree=None, random_state=None, ...)

In [138]:
pred_definition = DataDefinition(
    classification=[
        BinaryClassification(
            target="attrition",               # Nombre de la variable objetivo en tu data
            prediction_labels="prediction",   # Columna con predicción del modelo
            prediction_probas="prob",         # Columna con probabilidad (si la tienes)
            pos_label=1                       # Valor considerado como "positivo"
        )
    ],
    id_column="id_cliente",                   # ID único de cliente
    #timestamp="codmes_cliente",               # Columna de tiempo (formato YYYYMM)
    
    categorical_columns=[
        "attrition", "prediction", "flg_bancarizado", "rang_ingreso"
    ],
    
    numerical_columns=[
        "edad", "antiguedad", "prob","codmes_cliente"
    ]
)

In [139]:
df["attrition"] = df["attrition"].astype(str).astype("category")
df["flg_bancarizado"] = df["flg_bancarizado"].astype(str).astype("category")
df["rang_ingreso"] = df["rang_ingreso"].astype("category")
df["flag_lima_provincia"] = df["flag_lima_provincia"].astype("category")

df["tipo_requerimiento2"] = df["tipo_requerimiento2"].astype("category")
df["dictamen"] = df["dictamen"].astype("category")
df["producto_servicio_2"] = df["producto_servicio_2"].astype("category")
df["submotivo_2"] = df["submotivo_2"].astype("category")

df_ref = df[df["edad"] <= 30]
df_cur = df[df["codmes_cliente"] > 30]

In [140]:
FEATURES = ["flg_bancarizado", "rang_ingreso", "edad", "antiguedad"]
TARGET   = "attrition"

X_ref = df_ref[FEATURES]
y_ref = df_ref[TARGET].astype(int)

X_cur = df_cur[FEATURES]
y_cur = df_cur[TARGET].astype(int)

In [149]:

xgb = XGBClassifier(
        random_state=42,
        enable_categorical=True,   # ← aquí
        tree_method="hist"         # obligatorio con enable_categorical
)
xgb.fit(X_ref, y_ref)


for _df, _X in [(df_ref, X_ref), (df_cur, X_cur)]:
    _df["prediction"] = xgb.predict(_X)#.astype(str).astype("category")
    _df["prob"]       = xgb.predict_proba(_X)[:, 1].astype("float32")  # clase positiva


def cast_for_evidently(d):
    d["attrition"]        = d["attrition"]#.astype(str).astype("category")
    d["flg_bancarizado"]  = d["flg_bancarizado"].astype(str).astype("category")
    d["rang_ingreso"]     = d["rang_ingreso"].astype("category")
    d["edad"]             = d["edad"].astype("int16")
    d["antiguedad"]       = d["antiguedad"].astype("int16")
    return d

df_ref = cast_for_evidently(df_ref)
df_cur = cast_for_evidently(df_cur)

df_ref["attrition"] = df_ref["attrition"].astype(int)
df_ref["prediction"] = df_ref["prediction"].astype(int)
df_cur["attrition"] = df_cur["attrition"].astype(int)
df_cur["prediction"] = df_cur["prediction"].astype(int)


reference = Dataset.from_pandas(df_ref, data_definition=pred_definition)
current   = Dataset.from_pandas(df_cur, data_definition=pred_definition)

# DATA DRIFT DETECTION

In [150]:
from evidently.presets import DataDriftPreset
from monitoring_utils import detect_data_drift

In [152]:
# % of valid drifted columns (no IDs neither TIMESTAMPS) to consider the dataset drifted
drift_share = 0.3

## data drift

In [153]:
data_drift_report = Report([DataDriftPreset(drift_share=drift_share)], tags=["Data drift"])
data_drift_run = data_drift_report.run(current, reference)
data_drift_run.save_html("data_drift.html")
data_drift_ref = ws.add_run(project.id, data_drift_run)
print(f"Report's link: {data_drift_ref.url}")
print(f"Report's id: {data_drift_ref.id}")
#ws.delete_run(project.id, data_drift_ref.id)

/opt/conda/lib/python3.12/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning:

invalid value encountered in divide

/opt/conda/lib/python3.12/site-packages/numpy/lib/function_base.py:2898: RuntimeWarning:

invalid value encountered in divide

/opt/conda/lib/python3.12/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning:

invalid value encountered in divide

/opt/conda/lib/python3.12/site-packages/numpy/lib/function_base.py:2898: RuntimeWarning:

invalid value encountered in divide

/opt/conda/lib/python3.12/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning:

invalid value encountered in divide

/opt/conda/lib/python3.12/site-packages/numpy/lib/function_base.py:2898: RuntimeWarning:

invalid value encountered in divide

/opt/conda/lib/python3.12/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning:

invalid value encountered in divide

/opt/conda/lib/python3.12/site-packages/numpy/lib/function_base.py:2898: RuntimeWarning:

invalid value encount

Report's link: https://8kgtpeaxtj.us-east-1.awsapprunner.com/projects/0197c00c-c2c1-76a3-be64-3ee4604ef948/reports/0197c296-5c32-7bb0-8d13-1f207cf25f20
Report's id: 0197c296-5c32-7bb0-8d13-1f207cf25f20


In [154]:
detect_data_drift(data_drift_run)

True

# CONCEPT DRIFT DETECTION

In [156]:
from evidently.presets import ClassificationPreset
from monitoring_utils import get_metric_dict, get_metric
concept_drift_report = Report(
    [ClassificationPreset()],
    tags=["Concept drift"]
)

concept_drift_run = concept_drift_report.run(current, reference)
concept_drift_run.save_html("concept_drift.html")
concept_drift_ref = ws.add_run(project.id, concept_drift_run)
print(f"Report's link: {concept_drift_ref.url}")
print(f"Report's id: {concept_drift_ref.id}")

Report's link: https://8kgtpeaxtj.us-east-1.awsapprunner.com/projects/0197c00c-c2c1-76a3-be64-3ee4604ef948/reports/0197c296-8b5e-7cbe-8148-7fbf396d55b8
Report's id: 0197c296-8b5e-7cbe-8148-7fbf396d55b8


In [157]:
metric_dict = get_metric_dict(concept_drift_run)
metric_dict

{'Accuracy': 'e4f2ea2dab7ab3a65b16ef5124d02db4',
 'Precision': '3c2d79a01c4f6539c105bc7dff803faa',
 'Recall': '11c1894b301e560af740a01ee9739c2a',
 'F1Score': '2df7cb543daa4f42d1d5cdad5bb551ae',
 'RocAuc': '836d15857ce444b246783aad42a94779',
 'LogLoss': '275d2d67f1d9d892b74d8d9bdc07c233',
 'TPR': 'aede79b5b66270f86918a2111b694162',
 'TNR': '48b057cb0301dad5740a4a1a25050ffe',
 'FPR': '2e6d045688c97855c5661e6a8f6bd2e5',
 'FNR': 'd183f8fb51722080c04c08958a4579ff',
 'F1ByLabel': '960da7cc5ae6dfee13853a0b222466ca',
 'PrecisionByLabel': '57f465d9f518c33f7672e42e807a1737',
 'RecallByLabel': '66b666a06110f2e57883b2a7365fcf45',
 'RocAucByLabel': 'c1d27bd176ffbcc8c6b5871f2d88e6e6'}

In [158]:
cur_logloss, ref_logloss = get_metric(concept_drift_run, metric_dict["RocAuc"])
print(cur_logloss, ref_logloss)

0.671 0.79


In [159]:
delta = 0.04

In [160]:
if abs(ref_logloss-cur_logloss) > delta:
    if cur_logloss < ref_logloss:
        print("Significant concept drift detected")
else:
    print("No significant concept drift")

Significant concept drift detected


# ALERTING

In [108]:
import boto3

In [109]:
sns = boto3.client("sns")
response = sns.create_topic(Name="NewTopic")
topic_arn = response["TopicArn"]

In [110]:
endpoint = "germaingarcia17@gmail.com"

In [111]:
sns.list_subscriptions_by_topic(TopicArn=topic_arn)["Subscriptions"]
my_endpoint = [e["Endpoint"] for e in sns.list_subscriptions_by_topic(TopicArn=topic_arn)["Subscriptions"] if e["Endpoint"]==endpoint] 
if my_endpoint:
    print("Email already registered")
else:
    print("Registering email")
    sns.subscribe(
        TopicArn=topic_arn,
        Protocol="email",
        Endpoint=endpoint
    )

Registering email


In [114]:
if detect_data_drift(data_drift_run):
    sns.publish(TopicArn=topic_arn,
                Subject="Mensaje del GRUPO 5 Data drift detection Project MLOPS",
                Message=f"Check more details on the report: {data_drift_ref.url}"
    )